In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.6 The Fast Fourier Transform

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.6",
    title="The Fast Fourier Transform",
    blurb="From the discrete Fourier transform to the FFT: analysis and "
    "synthesis, aliasing and leakage, filtering, and why a violin and "
    "a saxophone playing the same note sound different.",
    difficulty="advanced",
    estimate="100–130 min",
)

## Notebook overview

The Fourier transform we met in analysis as an integral over all time; a
computer, with only a finite list of samples, uses its discrete cousin, the
**discrete Fourier transform** (DFT). The DFT is nothing more exotic than a
change of basis (rewriting a signal as a sum of pure complex exponentials), but
computing it naively costs $O(N^2)$, and the **fast Fourier transform** (FFT)
brings that down to $O(N\log N)$, a speed-up so consequential it reshaped
twentieth-century science and engineering. This notebook builds the DFT from
scratch, shows analysis and synthesis are an exact inverse pair, explains *why*
the FFT is fast, and then puts the transform to work: reading frequencies and
amplitudes off a spectrum, the sampling theorem and **aliasing**, **spectral
leakage** and windowing, and **filtering** by the convolution theorem.

The payoff is acoustic. The FFT of a sound's pressure waveform lays bare its
**harmonic spectrum**, and the *relative amplitudes* of the overtones are what
the ear hears as **timbre**: why a violin and a clarinet playing the same note
are instantly distinguishable. We will synthesize idealised string- and
clarinet-like tones and read their spectra, then connect the spectrum to the
instrument's geometry: a closed-open tube resonates at odd multiples of $c/4L$,
the hollow signature of the clarinet. (These are honest idealisations, not
recordings: real instruments add formants, transients, and conical-bore effects,
a caveat we keep in view.)

Everything here is a spectrum or a waveform: a still plot tells the whole story,
so there are no animations (a spectrogram of evolving timbre would be the place
for one, in a later applied notebook).

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: the hand-built DFT versus `np.fft.fft`, a known alias
> frequency, a resonance formula. A ✓ is strong evidence; a ✗ is a prompt to
> *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a signal-processing course. See Press et al.,
> *Numerical Recipes*, ch. 12–13 {cite}`numrecipes`; for the acoustics, Fletcher
> & Rossing, *The Physics of Musical Instruments* {cite}`fletcherrossing`.

## Theory in brief

### From Fourier series to the DFT

A periodic signal is a sum of harmonics: that is the Fourier series we know.
Sampling such a signal at $N$ equally spaced points and asking for the discrete
analogue gives the **discrete Fourier transform**: from samples $x_0,\dots,
x_{N-1}$ it produces

```{math}
:label: eq-dft
X_k = \sum_{n=0}^{N-1} x_n\,e^{-2\pi i\,kn/N}, \qquad k = 0,\dots,N-1.
```

Each $X_k$ is the overlap of the signal with one complex exponential of frequency
index $k$, so the DFT is a change of basis; in matrix form $\mathbf X = F\mathbf
x$ with $F_{kn}=e^{-2\pi i kn/N}$ the **DFT matrix**.

### The inverse, and the analysis/synthesis duality

The transform is exactly invertible: the **inverse DFT** rebuilds the signal,

```{math}
:label: eq-idft
x_n = \frac{1}{N}\sum_{k=0}^{N-1} X_k\,e^{+2\pi i\,kn/N},
```

so the forward transform is *analysis* (signal → its frequency content) and the
inverse is *synthesis* (frequency content → signal), and $\mathrm{ifft}\circ
\mathrm{fft}$ is the identity. (The one-line proof is the orthogonality of the
complex exponentials over the $N$ samples.) This duality is the conceptual heart, and it is
what makes filtering possible: edit the spectrum, then synthesize back.

### Why "fast": the FFT

Computed as written, {eq}`eq-dft` is a matrix–vector product, $O(N^2)$. The
**Cooley–Tukey FFT** instead splits the sum into its even- and odd-indexed terms,
each of which is itself a DFT of length $N/2$ that *shares* the same exponential
"twiddle" factors:

```{math}
:label: eq-fft
X_k = E_k + e^{-2\pi i k/N} O_k,
```

where $E_k,O_k$ are the half-length DFTs of the even and odd samples. Recursing
this split costs $O(N\log N)$: the saving is reusing the shared twiddle factors
rather than recomputing every $e^{-2\pi i kn/N}$.

### Bins, resolution, sampling, and leakage

With sampling rate $f_s$ and $N$ samples, bin $k$ sits at frequency $k f_s/N$, so
the resolution is $\Delta f = f_s/N$ (this is what `rfftfreq` returns; real
signals use `rfft`, which keeps only the non-redundant half by Hermitian
symmetry). Sampling has a hard limit, the **sampling theorem**:

```{math}
:label: eq-nyquist
\text{a signal is faithfully represented only up to } f_{\text{Nyq}} = f_s/2;
```

a frequency above $f_s/2$ is indistinguishable from (it **aliases** to) a lower
one (Press et al., *Numerical Recipes*, §12.1, state the theorem and its proof
idea; Exercise 5 demonstrates the folding directly). Separately, analysing a
finite stretch of a tone whose period does not fit a
whole number of times into the window smears its energy across neighbouring bins,
**spectral leakage**, which window functions (Hann, Hamming) suppress by tapering
the window's edges.

### The convolution theorem, and Parseval

Filtering rests on the **convolution theorem**: convolution in time is plain
multiplication in frequency,

```{math}
:label: eq-convtheorem
\widehat{(f * g)} = \hat f \cdot \hat g,
```

so to low-pass a signal one multiplies its spectrum by a mask and transforms back.
(Substituting {eq}`eq-dft` into the convolution sum proves it in a few lines;
Press et al., *Numerical Recipes*, §13.1, carry it out and build fast
convolution on it.) Finally, energy is the same whether counted in time or
frequency, because the change of basis is unitary up to the $1/N$ convention:
**Parseval's theorem**:

```{math}
:label: eq-parseval
\sum_{n=0}^{N-1} |x_n|^2 = \frac{1}{N}\sum_{k=0}^{N-1} |X_k|^2 .
```

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from ecp import validate

## Exercise 1 — The DFT from scratch

The cleanest way to meet the DFT is to build its matrix and multiply. The DFT
matrix has entries $F_{kn}=e^{-2\pi i kn/N}$, so $\mathbf X=F\mathbf x$ is exactly
{eq}`eq-dft`. We test it on two explicit length-8 signals: the unit impulse
$\mathbf x=[1,0,0,0,0,0,0,0]$, whose transform is a flat spectrum (every $X_k=1$,
since the impulse contains every frequency equally), and the pure tone
$x_n=\cos(2\pi\cdot 2n/8)$, whose energy should land entirely in bins $k=2$ and
$k=6$.

1. Build the DFT matrix and the matrix-product transform (the `dft_matrix` and
   `dft_naive` helpers below, `numpy.outer` for the exponent grid).
2. Transform the impulse and confirm the flat spectrum.
3. Transform the tone, confirm the energy sits in bins 2 and 6, and check both
   results against `numpy.fft.fft`.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    dft_naive(x_impulse),
    np.fft.fft(x_impulse),
    "the hand-built DFT matches np.fft.fft on the impulse",
    atol=1e-10,
)
validate.close(
    dft_naive(x_tone),
    np.fft.fft(x_tone),
    "the hand-built DFT matches np.fft.fft on cos(2π·2n/8)",
    atol=1e-10,
)

## Exercise 2 — The inverse transform and the analysis/synthesis duality

Because the DFT is just a change of basis, it undoes exactly, via the inverse DFT,
{eq}`eq-idft`. Taking the impulse signal $\mathbf x=[1,0,0,0,0,0,0,0]$ of
Exercise 1:

1. Transform it (`numpy.fft.fft`) and transform back (`numpy.fft.ifft`),
   confirming the round trip returns the original to machine precision.
2. Read each $X_k$ as the complex amplitude of one exponential: analysis followed
   by synthesis is the identity, and summing those exponentials *is* the signal.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.fft.ifft(np.fft.fft(x_impulse)),
    x_impulse,
    "ifft∘fft round-trips exactly",
    atol=1e-12,
)

## Exercise 3 — Why "fast": O(N log N)

The FFT's speed comes from the even/odd split of {eq}`eq-fft`, which we can write
as a short recursion (radix-2, for power-of-two $N$): split the samples into
even- and odd-indexed halves, transform each, and recombine with the twiddle
factors $e^{-2\pi i k/N}$.

1. Write the recursion (the `radix2_fft` helper) and check it against
   `numpy.fft.fft` on the impulse of Exercise 1.
2. Time `numpy.fft.fft` against the $O(N^2)$ matrix DFT of Exercise 1 across
   $N=2^4,\dots,2^{11}$ (`time.perf_counter`, averaged over repeats).
3. Watch the two scalings part ways against $N^2$ and $N\log N$ reference lines
   ({numref}`fig-fft-scaling`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    radix2_fft(x8),
    np.fft.fft(x8),
    "the recursive radix-2 FFT matches numpy",
    atol=1e-10,
)

## Exercise 4 — Reading a spectrum: frequencies and amplitudes

With the transform in hand, the everyday task is to read a real measurement.
Take the explicit two-tone signal

$$
x(t) = 3\sin(2\pi\cdot 50\,t) + 1.5\sin(2\pi\cdot 120\,t),
$$

sampled at $f_s=1000$ Hz for $T=1$ s (so $N=1000$ samples, resolution $\Delta f =
f_s/N = 1$ Hz).

1. Transform with `numpy.fft.rfft` and get the frequency axis from
   `numpy.fft.rfftfreq` (the right pair for a real signal).
2. Locate the two peaks.
3. Recover their amplitudes as $2|X_k|/N$ (the single-sided amplitude of a sine).
   Both frequencies (50 and 120 Hz) and amplitudes (3 and 1.5) should come back
   exactly, because each tone sits on an integer bin ({numref}`fig-fft-spectrum`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.sort(detected_freqs),
    [50.0, 120.0],
    "the two tones appear at 50 and 120 Hz",
    atol=1.0,
)
validate.close(
    amplitudes[np.argsort(detected_freqs)],
    [3.0, 1.5],
    "the peak amplitudes are recovered",
    rtol=1e-2,
)

## Exercise 5 — Aliasing and the sampling theorem

The sampling theorem, {eq}`eq-nyquist`, sets a hard ceiling: sampled at $f_s$, a
signal is faithful only below $f_s/2$, and anything higher folds down to a lower
frequency. Take the explicit tone $x(t)=\sin(2\pi\cdot 70\,t)$ and sample it at
$f_s=100$ Hz, whose Nyquist limit is only 50 Hz. The 70 Hz tone exceeds it and
aliases to $|70-100|=30$ Hz.

1. Sample the tone at $f_s=100$ Hz and locate the spectral peak: it *masquerades*
   as a 30 Hz tone, indistinguishable from the real thing in the samples
   ({numref}`fig-fft-alias`).
2. Sample the same tone at $f_s=1000$ Hz (Nyquist 500 Hz) and confirm it is
   recorded correctly at 70 Hz.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    apparent_freq, 30.0, "a 70 Hz tone sampled at 100 Hz aliases to 30 Hz", atol=1.0
)

## Exercise 6 — Spectral leakage and windowing

A subtler artefact appears when a tone's period does not fit a whole number of
times into the analysis window. Take the explicit tone $x(t)=\sin(2\pi\cdot
50.5\,t)$, sampled at $f_s=1000$ Hz for $T=1$ s: at 50.5 Hz the signal completes
$50.5$ periods in the window, falling *between* the 50 and 51 Hz bins, so its
energy smears across many bins: **spectral leakage**. Multiplying the signal by a
**Hann window** (a smooth taper to zero at both ends) concentrates the energy back
into the main lobe ({numref}`fig-fft-leakage`).

1. Build the tone and quantify its leakage: the fraction of spectral energy lying
   *outside* the two bins nearest 50.5 Hz (the `leakage` helper).
2. Apply a Hann window (`numpy.hanning`) and recompute.
3. Confirm the window reduces the leakage, at the cost of a slightly wider peak.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    leak_hann < leak_rect,
    "the Hann window reduces spectral leakage",
    f"leakage Hann {leak_hann:.3f} < rect {leak_rect:.3f}",
)

## Exercise 7 — Filtering via the transform (convolution theorem)

The convolution theorem, {eq}`eq-convtheorem`, makes filtering a one-liner in
frequency space: rather than convolve in time, edit the spectrum and transform
back. Take the explicit noisy signal

$$
x(t) = \sin(2\pi\cdot 5\,t) + 0.5\sin(2\pi\cdot 80\,t) + 0.3\,\eta(t),
$$

with $f_s=1000$ Hz, $T=2$ s, and noise $\eta$ drawn from
`numpy.random.default_rng(0).standard_normal` (one sample per time point). The
wanted signal is the 5 Hz tone; the 80 Hz tone and the broadband noise are not.

1. Transform with `numpy.fft.rfft` and zero every bin above a $20$ Hz cutoff.
2. Inverse-transform with `numpy.fft.irfft`.
3. Confirm the 80 Hz component (and most of the noise) has vanished
   ({numref}`fig-fft-filter`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    amp80_after < 0.05 * amp80_before,
    "zeroing high-frequency bins and inverse-transforming removes the 80 Hz tone",
    f"80 Hz amplitude {amp80_before:.3f} → {amp80_after:.2e}",
)

## Exercise 8 — Timbre: the harmonic spectrum of a note

Now the acoustics. Two instruments playing the same pitch sound different because
their overtones have different *relative amplitudes*: that is timbre, and the FFT
reads it directly. Synthesize two explicit half-second notes at the pitch
$f_0=220$ Hz (A3), sampled at $f_s=44100$ Hz: a **string/violin-like** tone with
*all* harmonics,

$$
x_\text{violin}(t) = \sum_{n=1}^{19} \tfrac1n \sin(2\pi n f_0 t),
$$

and a **clarinet-like** tone with only the *odd* harmonics,

$$
x_\text{clarinet}(t) = \sum_{n\ \text{odd},\,1}^{19} \tfrac1n \sin(2\pi n f_0 t).
$$

1. Synthesize both tones.
2. Compute their spectra (`numpy.fft.rfft`) and read off the harmonic amplitudes.
3. Confirm the signature difference: the clarinet's even harmonics are absent
   while the violin's are all present ({numref}`fig-fft-timbre`).

These are deliberate **idealisations** (real
notes carry formants, transients, and bore-shape effects; a saxophone's conical
bore sounds a full harmonic series despite being closed-open), but the
odd-versus-all contrast is the essential lesson.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    even_clar.max() < 1e-6 and even_viol.min() > 0.01,
    "the clarinet-like tone has only odd harmonics; the violin-like tone has all of them",
    f"clarinet even max {even_clar.max():.1e}, violin even min {even_viol.min():.3f}",
)

## Exercise 9 — Acoustic resonances of a bore (student exercise)

Where do those harmonics come from? An instrument sounds the frequencies its air
column resonates at, and the geometry fixes them. For a **closed-open cylindrical
tube** (closed at the mouthpiece, open at the bell, the clarinet's idealisation)
of length $L=0.66$ m with sound speed $c=343$ m/s, the resonances are the *odd*
multiples

$$
f_n = (2n-1)\,\frac{c}{4L}, \qquad n = 1, 2, 3, \dots,
$$

which is exactly why the clarinet's spectrum in Exercise 8 had only odd harmonics.

1. Compute the first four resonances and relate them to that FFT spectrum.
2. Contrast with an **open-open tube** (a flute-like idealisation), whose
   resonances $f_n = n\,c/2L$ form the *full* harmonic series.
3. Connect the two: the bore's resonance peaks select which harmonics the
   instrument can sound — the bridge from geometry to timbre.

There is no animation here; this is a resonance calculation. *(A personal aside: I have always found it striking that one can hear
the difference between a clarinet and a flute and that the reason is, at bottom,
one boundary condition (open versus closed) turned into a factor of two.)*

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    f_closed_open,
    [(2 * n - 1) * c / (4 * L) for n in range(1, 5)],
    "closed-open tube resonances are odd multiples of c/4L",
    rtol=1e-6,
)

## Exercise 10 — Power spectrum and Parseval's theorem

A useful consistency check closes the loop: energy is conserved between the time
and frequency domains, **Parseval's theorem** of {eq}`eq-parseval`.

1. For the two-tone signal of Exercise 4 ($f_s=1000$ Hz, $1$ s), compute the
   time-domain energy $\sum_n |x_n|^2$ and the frequency-domain energy
   $\tfrac1N\sum_k |X_k|^2$ (`numpy.fft.fft`).
2. Confirm they agree to rounding.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    time_energy,
    freq_energy,
    "Parseval: energy is conserved between time and frequency domains",
    rtol=1e-10,
)

## Notebook summary

- The DFT from scratch and its analysis/synthesis duality, then the $O(N\log N)$ FFT; reading
  a spectrum's frequencies and amplitudes; **aliasing** and the sampling theorem; spectral
  leakage and windowing.
- The convolution theorem for filtering; the harmonic spectrum behind timbre; acoustic bore
  resonances; and Parseval's theorem tying the power spectrum to the signal energy.

## Outlook

- **The 2-D FFT** filters and compresses images, the spectral cousin of the SVD
  compression of [§0.5](eigenvalues-svd.ipynb): both discard small coefficients.
- **Spectral differentiation** computes a derivative by multiplying by $ik$ in
  frequency space, the seed of spectral methods for PDEs.
- **The short-time Fourier transform (spectrogram)** tracks how timbre evolves
  (a vibrato, a note's attack) and is the natural place for a genuine *animation*
  in a later applied notebook.
- **In physics**, the FFT computes structure factors and diffraction patterns,
  correlation functions, and the spectral solution of field equations.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()